[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aarondomenzain/tracking-softmatter-aarond/blob/tracking-softmatter-aarond-dev/tutorial/tracking/tracking_spheres.ipynb)

# Particle Tracking Tutorial: Trajectory Linking with Organic Fluorophores

In this tutorial, you’ll explore different methods to link particle localizations across time to recostruct trajectories using both simulated data and real experimental videos.

You’ll start by generating a simulated movie of fluorophores labeling proteins of the cell membrane undergoing stochastic motion, mimicking a real biophysics experiment. For each frame, you'll use LodeSTAR, a self-supervised neural network, to detect and localize fluorophores. Then comes the core challenge: linking localization into trajectories.

Here you’ll test and compare various methods to perform the linking of the trajectories:

- Nearest-neighbor linking (using TrackPy — a classic in particle tracking)

- Linear Assignment Problem (LAP) (using LapTrack - a more flexible and general framework)

- MAGIK (a geometric deep learning method based on graph neural networks)

You'll evaluate **end-to-end tracking performance**, including detection, linking, and trajectory-level evaluation. You’ll be using Python libraries like NumPy, SciPy, Matplotlib, scikit-image, PyTorch, DeepTrack, and Deeplay. 

## Table of Contents

0. [Importing the Required Libraries and Loading Utility Functions](#importing-the-required-libraries-and-loading-utility-functions)
1. [Loading and Visualizing Experimental Videos](#loading-and-visualizing-experimental-videos)
2. [Simulating Realistic Videos with DeepTrack](#simulating-realistic-videos-with-deeptrack)
    - [Simulating a Single Fluorophore](#simulating-a-single-fluorophore)
    - [Simulating a Video Frame](#simulating-a-video-frame)
    - [Simulating Brownian Trajectories](#simulating-brownian-trajectories)
    - [Simulating a Video](#simulating-a-video)

2. [Detecting and Localizing Particles with LodeSTAR](#detecting-and-localizing-particles-with-lodestar)
    - [Training LodeSTAR with Simulations](#training-lodestar-with-simulations)
    - [Evaluating LodeSTAR on Simulations](#evaluating-lodestar-on-simulations)
    - [Applying LodeSTAR to Simulations](#applying-lodestar-to-simulations)
    - [Applying LodeSTAR to Experiments](#applying-lodestar-to-experiments)

3. [Method 1: Nearest-neighbor Linking with TrackPy](#method-1-nearest-neighbor-linking-with-trackpy)
    - [Linking Localizations in Simulations](#linking-localizations-in-simulations)
    - [Evaluating Linking Performance](#evaluating-linking-performance)
    - [Linking Localizations in Experiments](#linking-localizations-in-experiments)

4. [Method 2: Linear Assignment Problem (LAP) with LapTrack](#method-2-linear-assignment-problem-lap-with-laptrack)
    - [Linking Localizations in Simulations](#linking-localizations-in-simulations)
    - [Evaluating Linking Performance](#evaluating-linking-performance)
    - [Linking Localizations in Experiments](#linking-localizations-in-experiments)

5. [Method 3: MAGIK](#method-3-magik)
    - [Training MAGIK with Simulations](#training-magik-with-simulations)
    - [Evaluate MAGIK on a test video](#evaluate-magik-on-a-test-video)
    - [Linking Localizations in Simulations](#linking-localizations-in-simulations)
    - [Evaluating Linking Performance](#evaluating-linking-performance)
    - [Linking Localizations in Experiments](#linking-localizations-in-experiments)



## Importing the Required Libraries and Loading Utility Functions

Uncomment the next cell if running on Google Colab/Kaggle.

In [ ]:
#!pip install deeptrack deeplay trackpy laptrack -q

In [ ]:
# Standard libraries.
import logging
import os
import random
import sys

# Configuration
import matplotlib
matplotlib.rcParams["animation.embed_limit"] = 60  # Larger animations inline
logging.disable(logging.WARNING)  # Suppress warnings and below

# Core Scientific Stack
import numpy as np
import pandas as pd

# Plotting and Display
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

# Machine Learning
import deeplay as dl
from sklearn.metrics import f1_score
import torch
from torchvision.transforms import Compose
from torch_geometric.loader import DataLoader
from torchmetrics import MetricCollection


# Particle Tracking and Simulation
import deeptrack as dt
from laptrack import LapTrack
import trackpy as tp

Load a set of custom functions defined specifically for this notebook from the `utils` directory. For detailed documentation of each function, refer to the comments and docstrings within the files.

In [ ]:
# Load functions and utilities for dataset generation and visualization.
# Sys path modification to import utils from parent folder.
sys.path.append(os.path.abspath(os.path.join("..", "..")))

# Import all the functions contained in the folder utils.
import utils

Set random seeds to make results reproducible across runs, especially during training and data simulation. Also select the best device for computations with Torch.

In [ ]:
# Set a fixed seed value.
seed = 2134

# Python, NumPy, and PyTorch (CPU).
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)

# Only set CUDA seeds if a GPU is available.
if torch.cuda.is_available():
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

print(f"Seeds set to {seed} (with CUDA: {torch.cuda.is_available()})")

# Get the best available device for Torch computation.
if torch.cuda.is_available():
    device = torch.device("cuda")
    print(f"Using CUDA GPU: {torch.cuda.get_device_name(0)}")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
    print("Using Apple GPU (MPS)")
else:
    device = torch.device("cpu")
    print("Using CPU")

## Loading and Visualizing Experimental Videos

You'll use an experimental video of fluorophores labeling membrane proteins in living cells, recorded with fluorescence microscopy. Data provided by the Quantitative BioImaging lab (https://mon.uvic.cat/qubilab/).


In [ ]:
# Define the folder and video file name.
video_folder = "videos"
video_file_name = "experimental_video.npy"

# Construct the full path.
video_path = os.path.join(video_folder, video_file_name)

# Load the video data.
exp_video = np.load(video_path)

utils.play_video(exp_video, "Experimental Video")

Select the first frame of the video.

In [ ]:
# Select the first frame of the video.
exp_image = exp_video[0]

# Get the shape of the image.
assert exp_image.shape[0] == exp_image.shape[1], "Warning: Image not square!"
exp_image_size = exp_image.shape[0]

Manually select a single fluorophore by specifying its centroid coordinates (x, y) and a box width.

In [ ]:
# Box size to zoom in an individual particle.
exp_crop_size = 15

# Coordinates of the center of the particle.
x_center = 53
y_center = 76

Display the first frame of the video together with the selected fluorophore.

In [ ]:
# # Calculate top-left corner of the crop.
x = x_center - exp_crop_size // 2
y = y_center - exp_crop_size // 2

# Select a crop as a subset of the entire image.
exp_crop = exp_image[
    y:y + exp_crop_size,  # row (y)
    x:x + exp_crop_size,  # column (x)
]

# Initialize figure instance.
fig = plt.figure()

vmin, vmax = np.percentile(exp_image, [1, 99])

# Draw a red rectangle around the crop.
fig.add_subplot(111)
plt.imshow(exp_image, cmap="gray", vmin=vmin, vmax=vmax)
plt.title("Experimental Image", size=13)
plt.plot([x, x, x + exp_crop_size, x + exp_crop_size, x],
         [y, y + exp_crop_size, y + exp_crop_size, y, y], 'r-')
plt.axis("off")

# Plot the rectangle on the top right corner.
fig.add_subplot(666)
plt.imshow(exp_crop, cmap="gray", vmin=vmin, vmax=vmax)
plt.axis("off")
plt.show()

## Simulating Realistic Videos with DeepTrack

DeepTrack allows you to simulate physically realistic microscopy images and videos, enabling precise control over imaging parameters and particle properties. These simulations provide ground-truth data, making them ideal for benchmarking classical and AI-based tracking methods, as well as for training neural networks in a controlled environment.

### Simulating a Single Fluorophore

Adjust the simulation parameters to accurately replicate the features observed in the cropped region of the experimental image.

In [ ]:
# Same as the box width.
sim_crop_size = exp_crop_size  

# Size of a pixel in nanometers in the output image.
pixel_size_nm = 110 # In nm.

# Radius of the particle.
particle_radius = 2.5 # In nm.

# Define central spherical scatterer.
# Note: Particle is diffraction limited, you can alternative use dt.PointParticle().
sphere = dt.Sphere(
    position=0.5 * np.array([sim_crop_size, sim_crop_size]) -(0.5,0.5),
    z= 230 * dt.units.nm, # Particle in focus.
    radius= particle_radius * dt.units.nm,  # Radius in nm
    intensity= 1.6E3,  # Field magnitude squared
)

# Simulate the properties of the fluorescence microscope.
optics = dt.Fluorescence(
    NA=1.47,  # Numerical aperture
    wavelength=580 * dt.units.nm,
    refractive_index_medium=1.33,
    output_region=[0, 0, sim_crop_size, sim_crop_size],
    magnification=1.0,
    resolution=pixel_size_nm * dt.units.nm,  # Camera effective resolution
)

# Apply transformations.
sim_crop = (
    optics(sphere)
    >> dt.Background(117)  # Background intensity level
    >> dt.Poisson(snr=550)  # Signal-to-noise ratio (SNR) of the image
)

# Turn the crop into a NumPy array.
sim_crop = np.squeeze(sim_crop())

# Plot the simulated and experimental crops.
fig, axes = plt.subplots(1, 2)

# Simulated crop.
plot = axes[0].imshow(sim_crop, cmap="gray")
axes[0].axis("off")
axes[0].set_title("Simulated Crop")

# Experimental crop.
axes[1].imshow(exp_crop, cmap="gray")  
axes[1].axis("off")
axes[1].set_title("Experimental Crop")

# Adjust layout and show plot.
plt.tight_layout()
plt.show()

Extract and visualize the raw intensity profiles along the central horizontal and vertical lines of both the simulated and experimental crops. This comparison helps evaluate how well the simulation reproduces the intensity distribution observed in real microscopy images.

In [ ]:
# Compute the center index.
center = sim_crop_size // 2

# Extract horizontal (row) profiles.
sim_horiz = sim_crop[center, :]
exp_horiz = exp_crop[center, :]

# Extract vertical (column) profiles.
sim_vert = sim_crop[:, center]
exp_vert = exp_crop[:, center]

# Create a 1×2 subplot.
fig, axes = plt.subplots(1, 2, figsize=(12, 4), tight_layout=True,sharey=True)

# --- Horizontal profile ---
axes[0].plot(sim_horiz, label="Simulated Crop", color="orange")
axes[0].plot(exp_horiz, label="Experimental Crop", color="blue")
axes[0].set_xlabel("Pixel (x)")
axes[0].set_ylabel("Intensity")
axes[0].set_title("Horizontal Intensity Profile (Center Row)")
axes[0].legend()
axes[0].grid(True, linestyle="--", alpha=0.5)

# --- Vertical profile ---
axes[1].plot(sim_vert, label="Simulated Crop", color="orange")
axes[1].plot(exp_vert, label="Experimental Crop", color="blue")
axes[1].set_xlabel("Pixel (y)")
axes[1].set_ylabel("Intensity")
axes[1].set_title("Vertical Intensity Profile (Center Column)")
axes[1].legend()
axes[1].grid(True, linestyle="--", alpha=0.5)

### Simulating a Video Frame

Create a simulated image containing non-overlapping spherical particles. Begin by generating their coordinates to serve as the ground-truth positions. Then, use DeepTrack to render optically realistic particles at these coordinates, resulting in a physically plausible microscopy image.

In [ ]:
# Parameters of the simulation.
sim_image_size = 128
N_particles = 40

# Dictionary for particle properties. Dimensions are set with lambda
# functions to introduce variety to the dataset.
particle_properties = {
    "intensity": lambda: np.random.uniform(1.5, 1.6) * 1E3,
    "z": lambda: np.random.uniform(-150, 250) * dt.units.nm,
    "radius": particle_radius * dt.units.nm,
}

# Set the optical properties of the microscope.  This dictionary is a DeepTrack
# optics object.
optics_properties = dt.Fluorescence(
    NA=1.47,  # Numerical aperture
    wavelength=580 * dt.units.nm,
    refractive_index_medium=1.33,
    output_region=[0, 0, sim_image_size, sim_image_size],
    magnification=1.5,
    resolution=pixel_size_nm * dt.units.nm,  # Camera effective resolution
)

# Generate ground truth positions. 
sim_gt_pos = utils.generate_centroids(
    num_particles=N_particles,
    fov_size=sim_image_size,
    particle_radius=particle_radius,
)
# Simulate image.
sim_image = utils.transform_to_video(
    sim_gt_pos,
    fov_size=sim_image_size,
    core_particle_props=particle_properties,
    optics_props=optics_properties,
    background_props={"peak_snr": 200, "background_mean": 117, "background_std": 3.5},
)


Visualize the simulated image and compare it with the experimental one.

In [ ]:
# Plot the simulated and experimental images.
fig, axes = plt.subplots(1, 2)
vmin, vmax = np.percentile(exp_image, [1, 99])

# Simulated image.
plot = axes[0].imshow(sim_image, cmap="gray", vmin=vmin, vmax=vmax)
axes[0].axis("off")
axes[0].set_title("Simulated Image")

# Experimental image.
axes[1].imshow(exp_image, cmap="gray", vmin=vmin, vmax=vmax)  
axes[1].axis("off")
axes[1].set_title("Experimental Image")

# Adjust layout and show plot.
plt.tight_layout()
plt.show()

Perform a more quantitative comparison by plotting the intensity histograms.

In [ ]:
# Flatten the arrays to 1D.
sim_vals = sim_image.ravel()
exp_vals = exp_image.ravel()

# Compute common bin edges.
all_vals = np.concatenate([sim_vals, exp_vals])
num_bins = 50
bins = np.linspace(all_vals.min(), all_vals.max(), num_bins + 1)

# Create figure with two subplots sharing axes.
fig, axes = plt.subplots(
    1, 2, 
    figsize=(10, 4), 
    sharey=True, 
    sharex=True, 
    tight_layout=True
)

# Histogram for simulated image.
axes[0].hist(sim_vals, bins=bins, alpha=0.7, edgecolor="black")
axes[0].set_title("Simulated Image Histogram")
axes[0].set_yscale("log")
axes[0].set_xlabel("Intensity")
axes[0].set_ylabel("Pixel Count")

# Histogram for experimental image.
axes[1].hist(exp_vals, bins=bins, alpha=0.7, edgecolor="black")
axes[1].set_title("Experimental Image Histogram")
axes[1].set_yscale("log")
axes[1].set_xlabel("Intensity")

plt.show()

### Simulating Brownian Trajectories

You'll simulate a set of trajectories that visually resemble the experimental data to evaluate the performance of different tracking methods. The goal is to replicate the stochastic motion of labeled poteins as observed in the experimental videos. This is done using the `simulate_Brownian_trajs()` function from the utility file, which generates 2D trajectories based on a random walk model. Since protein motion is rather heterogeneous, trajectories are simulated with a range of diffusion coefficients. Trajectories that move out and back in the field of view due to boundary conditions are treated as separate trajectories. Refer to the function for more details.

In [ ]:
# Simulation parameters.
number_particles = 35
number_timesteps = 50

# Simulate trajectories for one video.
sim_trajs_gt = utils.simulate_Brownian_trajs(
    num_particles=number_particles,
    num_timesteps=number_timesteps,
    fov_size=sim_image_size,
    diffusion_std=(0.1, 1.0),  # Corresponds to sqrt(2Dt)
)

Trajectories blink and appear/disappear during the movie, let's reproduce that in simulations using a function that add some stochasticity. When off, particle positions are replaced with NaNs.

In [ ]:
sim_trajs_gt = utils.apply_blinking(sim_trajs_gt)

For evaluation purposes, the trajectories are transformed into a list and NaNs are removed.

In [ ]:
# Create list of trajectories.
sim_trajs_gt_list = utils.trajs_array_to_list(
    trajs=sim_trajs_gt,
)

### Simulating a Video

You'll now simulate a video of particle motion that resembles experimental. You'll play the two video simultaneously for comparison.

In [ ]:
# Define whether to force simulations even if video exists.
force_simulation = True

# Define folder and path for storing simulated video.
folder_name = "./videos"
simulation_path = os.path.join(folder_name, "simulated_video.npy")

if not os.path.isfile(simulation_path) or force_simulation:
    # Simulate video.
    sim_video = utils.transform_to_video(
        np.delete(sim_trajs_gt, 2, 2),  # Remove frame axis
        fov_size=sim_image_size,
        core_particle_props=particle_properties,
        optics_props=optics_properties,
        background_props={"peak_snr": 100, "background_mean": 117, "background_std": 3.5},
        save_video=True,
        path=simulation_path,
    )
    print(f"Saved simulated video to '{simulation_path}'.")
else:
    # Load previously simulated video.
    sim_video = np.load(simulation_path)
    print(f"Loaded simulated video from '{simulation_path}.npy'.")

# Play both simulated and experimental videos and compare.
utils.play_video(sim_video, "Simulated Video")
utils.play_video(exp_video, "Experimental Video")

## Detecting and Localizing Particles with LodeSTAR

LodeSTAR will be used to detect the fluorophores' positions at each frame of the video, similarly as shown in the **Detections** notebooks of this tutorial. These positions will be passed to different linking methods to build trajectories and compare their performance.

### Training LodeSTAR with Simulations

For the training, you'll generate a set of synthetic crops, each containing a single fluorophore. These are produced using a mix of simulation and data augmentation. The particle’s size and intensity are randomized within realistic bounds, making the dataset more diverse and representative of what you'd expect in actual microscopy data.

In [ ]:
# Number of samples, image size, and particles.
num_samples = 256
train_crops_size = sim_crop_size
max_num_particles = 1
force_simulation = True  # Flag to force simulation even if data exists

# Create path to store training dataset.
folder_name = "LodeSTAR"
training_dataset_filename = "LodeSTAR_training_dataset_fluorophores.npz"
training_dataset_folder = os.path.join(folder_name, "training_data")
training_dataset_filepath = os.path.join(
    training_dataset_folder, 
    training_dataset_filename,
)

# Create the enclosing directory if not existent already.
os.makedirs(training_dataset_folder, exist_ok=True)

# Dictionary for particle properties. Dimensions are set with lambda
# functions to introduce variety to the dataset.
particle_properties = {
    "intensity": lambda: np.random.uniform(1.0, 1.6) * 1E3,
    "z": lambda: np.random.uniform(-500, 500) * dt.units.nm,
    "radius": particle_radius * dt.units.nm,
}

# Set the optical properties of the microscope.  
optics_properties = dt.Fluorescence(
    NA=1.47,  # Numerical aperture
    wavelength=580 * dt.units.nm,
    refractive_index_medium=1.33,
    output_region=[0, 0, sim_crop_size, sim_crop_size],
    magnification=lambda: np.random.uniform(1.0, 2.0),
    resolution=pixel_size_nm * dt.units.nm,  # Camera effective resolution
)

# Try to load preexisting simulated data; if not available or forced, raise an
# exception error to generate new data.
try:
    if force_simulation: 
        # Raise the exception error if simulation is forced.
        raise FileNotFoundError("Forced simulation by user request.")
    
    if not os.path.isfile(training_dataset_filepath):
        # If file is not found, start training.    
        raise FileNotFoundError(
            "Training dataset file not found. Starting simulation."
        )
    
    # Load existing data
    data = np.load(training_dataset_filepath)
    crop_images = data["crop_images"]
    crop_maps = data["crop_maps"]
    Nsamples = len(crop_images)  # Update number of samples.
    print(f"Loaded file: {training_dataset_filepath}")
        
# Handle the case of either file not found or forced training.
except FileNotFoundError:
    
    # Perform simulation if file not found or forced.
    crop_images, crop_maps = utils.generate_particle_dataset(
        num_samples=num_samples,
        fov_size=train_crops_size,
        max_num_particles = max_num_particles,
        core_particle_dict=particle_properties,
        optics_properties=optics_properties,
        background_props={"peak_snr": 100, "background_mean": 117, "background_std": 3.5},
    )

    # Save the simulated training dataset.
    np.savez(
        training_dataset_filepath, 
        crop_images=crop_images, 
        crop_maps=crop_maps,
    )
    print("New dataset generated and saved.")
    
# Plot up to 18 simulated crops.
utils.plot_crops(np.squeeze(crop_images), title="Simulated Crops")

Build a training pipeline that randomly selects and augments crops on the fly. Each crop is chosen randomly from the simulated dataset and normalized with slightly varying min/max values to mimic contrast and illumination changes.

This setup creates a diverse training set without needing to store multiple augmented copies.

In [ ]:
# Define the training pipeline with additional settings, such as multiplication
# of global intensity and Gaussian noise.
selector = dt.Lambda(
    lambda i: lambda x: x[i],
    i=lambda l: np.random.randint(l),
    l=len(crop_images),
)

# Define a training pipeline with augmentations. 
training_pipeline = (
    dt.Value(lambda: crop_images)
    >> dt.NormalizeMinMax(0.0, 1.0)
    >> dt.Gaussian(0.0, 0.025)
    >> dt.NormalizeMinMax(
        lambda: np.random.uniform(0.0, 0.1), 
        lambda: np.random.uniform(0.9, 1.0)
    )
    >> selector
)

# Build a training dataset with a pipeline that randomly selects a crop each
# time.
train_dataset = dt.pytorch.Dataset(
    training_pipeline,
    length=256,
)

# Dataloader contains the randomized training dataset and additional settings.
dataloader = dl.DataLoader(
    train_dataset,
    batch_size=12,
)

# We can check one of the elements in the training pipeline with augmentations.
sanity_check_crop = training_pipeline.update().resolve()

plt.figure()
plt.imshow(np.squeeze(sanity_check_crop), cmap="gray", aspect="equal")
plt.title("Random Crop from Training Pipeline")
plt.axis("off")
plt.show()

Create a LodeSTAR model with 4 random geometric transformations per input and set up the trainer, which will handle the training loop. It runs for 50 epochs.

In [ ]:
# Initiate LodeSTAR.
lodestar = dl.LodeSTAR(
    n_transforms=4,
    optimizer=dl.AdamW(lr=1e-3),
).build()

# Set up the trainer and specify number of epochs.
trainer_lodestar = dl.Trainer(max_epochs=50, accelerator="auto")

Check whether pre-trained LodeSTAR weights already exist on disk. If they are missing or if training is explicitly forced, the model is trained on the experimental crops using the specified training pipeline, and the resulting weights are saved for future use.

If the weights are already available and training is not forced, they are simply loaded from file.

Afterward, the model is set to evaluation mode, which disables training-specific behaviors, ensuring consistent behavior during inference.

In [ ]:
# Define whether to force training even if weights exist.
force_training = True

# Define folder and path for storing LodeSTAR experimental weights.
folder_name = "LodeSTAR"
lodestar_path = os.path.join(folder_name, "lodestar_weights")

# Train or load weights.
if not os.path.isfile(lodestar_path) or force_training:
    print("Training LodeSTAR on experimental data "
          "(either forced or weights not found).")
    
    # Ensure the save directory exists.
    os.makedirs(folder_name, exist_ok=True)

    # Train the model.
    trainer_lodestar.fit(lodestar, dataloader)
    
    # Plot training history.
    trainer_lodestar.history.plot() 

    # Save weights.
    torch.save(lodestar.state_dict(), lodestar_path)
    print(f"Saved experimental LodeSTAR weights to '{lodestar_path}'.")
else:
    # Load pre-trained weights.
    lodestar.load_state_dict(torch.load(lodestar_path, weights_only=True))
    print(f"Loaded preexisting LodeSTAR weights from '{lodestar_path}'.")

# Set model to evaluation mode.
lodestar.eval();

# Move the model to the best available device (optional).
lodestar.to(device);

### Evaluating LodeSTAR on Simulations

Apply the trained LodeSTAR model to a frame of the simulated video.  Set the inference parameters that control detection sensitivity. Get the prediction features, which can be useful for visualizing detections. Extract the final coordinates of the detected particles and print how many particles were detected. Plot the predicted localizations from LodeSTAR alongside the ground truth on the simulated image and quantify the performance.

In [ ]:
# Select frame number.
frame_index = 20

# Format according to the (N,C,X,Y) convention.
sim_frame_formatted = utils.format_image(sim_video[frame_index])

# Parameters for inference.
alpha = 0.9
beta = 1 - alpha
cutoff = 0.95

# Get localizations from experimental image.
sim_locs_pred = lodestar.detect(
    sim_frame_formatted, 
    alpha=alpha,
    beta=beta,
    mode="quantile",
    cutoff=cutoff,
)[0]

# Print detection number for reference.
print(f"Found {len(sim_locs_pred[:,1])} detections.")

utils.plot_predicted_positions(
   image=sim_frame_formatted[0, 0, :, :],
   pred_positions=sim_locs_pred,
   gt_positions=sim_trajs_gt[frame_index],
   title="Lodestar Trained on Simulated Crops - Simulated Image",
)

#  Evaluate performance.
utils.evaluate_locs(
    sim_locs_pred, sim_trajs_gt[frame_index], distance_th=1.5*pixel_size_nm,
);

### Applying LodeSTAR to Simulations

Iteratively apply LodeSTAR to every frame of the simulated video and store localizations in a dataframe.

In [ ]:
# Define a dataframe to store all the LodeSTAR detections.
df_sim_video = []
for frame_index, sim_frame in enumerate(sim_video):

    # Format image for LodeSTAR: shape must be (N, C, X, Y).
    sim_frame_formatted = utils.format_image(sim_frame)

    # Get detections from LodeSTAR.
    sim_locs_pred = lodestar.detect(
        sim_frame_formatted,
        alpha=alpha,
        beta=beta,
        cutoff=cutoff,
        mode="quantile",
    )[0]

    # Store detections in a DataFrame.
    df_frame = pd.DataFrame(sim_locs_pred, columns=["x", "y"])
    df_frame["frame"] = frame_index
    df_sim_video.append(df_frame)

    # Print no. of detections every 10 frames.
    if frame_index % 10 == 0:
        print(f"Detections in frame {frame_index}: {len(sim_locs_pred)}")

# Combine all detections into a single DataFrame.
df_sim_video = pd.concat(df_sim_video, ignore_index=True)

### Applying LodeSTAR to Experiments

Apply the trained LodeSTAR model to a frame of the experimental video to check that the inference parameters produce sensible detections by visual inspection.

In [ ]:
# Select frame number.
frame_index = -1

# Format according to the (N,C,X,Y) convention.
exp_frame_formatted = utils.format_image(exp_video[frame_index])

# Parameters for inference.
alpha = 0.9
beta = 1 - alpha
cutoff = 0.95

# Get localizations from experimental image.
exp_locs_pred = lodestar.detect(
    exp_frame_formatted, 
    alpha=alpha,
    beta=beta,
    mode="quantile",
    cutoff=cutoff,
)[0]

# Print detection number for reference.
print(f"Found {len(exp_locs_pred[:,1])} detections.")

utils.plot_predicted_positions(
   image=exp_frame_formatted[0, 0, :, :],
   pred_positions=exp_locs_pred,
   title="Lodestar Trained on Simulated Crops - Experimental Image",
)

Iteratively apply LodeSTAR to every frame of the experimental video and store localizations in a dataframe.

In [ ]:
# Define a dataframe to store all the LodeSTAR detections.
df_exp_video = []
for frame_index, exp_frame in enumerate(exp_video):

    # Format image for LodeSTAR: shape must be (N, C, X, Y).
    exp_frame_formatted = utils.format_image(exp_frame)

    # Get detections from LodeSTAR.
    exp_locs_pred = lodestar.detect(
        exp_frame_formatted,
        alpha=alpha,
        beta=beta,
        cutoff=cutoff,
        mode="quantile",
    )[0]

    # Store detections in a DataFrame.
    df_frame = pd.DataFrame(exp_locs_pred, columns=["x", "y"])
    df_frame["frame"] = frame_index
    df_exp_video.append(df_frame)

    # Print no. of detections every 10 frames.
    if frame_index % 10 == 0:
        print(f"Detections in frame {frame_index}: {len(exp_locs_pred)}")

# Combine all detections into a single DataFrame.
df_exp_video = pd.concat(df_exp_video, ignore_index=True)

## Method 1: Nearest-Neighbor Linking with TrackPy

TrackPy constructs trajectories by linking localized positions across frames using a predictive nearest-neighbor algorithm. The input must be a Pandas DataFrame containing the particle positions, usually with columns `x`, `y`, and `frame`.

See the [TrackPy tutorial on prediction and linking](https://soft-matter.github.io/trackpy/dev/tutorial/prediction.html) for more details on how the algorithm works and how to tune parameters like `search_range` and `memory`.

### Linking Localizations in Simulations

Linking typically depends on two parameters: a search radius and a maximum number of missed frames allowed for reconnection.
We define them here and use them consistently throughout the tutorial, adapting them to the different situations.

In [ ]:
max_search_radius = 5  # maximum allowed displacement between frames (pixels)
max_missed_frames = 3   # maximum number of consecutive missed frames for gap closing

Apply the method to the localization dataframe.

In [ ]:
# Link detections across frames into trajectories using trackpy.link().
# The `search_range` parameter sets the maximum allowed displacement (in pixels)
# between frames, and the `memory` parameter specifies how many consecutive frames
# a particle may be missing and still be linked to the same trajectory.
sim_trajs_pred_method1 = tp.link(
    df_sim_video,
    search_range=max_search_radius,
    memory=max_missed_frames,
)

Create a trajectory list from the output dataframe.

In [ ]:
# Minimum number of distinct frames required to keep a trajectory.
min_trajectory_length = 15

# Group detections by particle. Each group represents one trajectory.
# `sort=False` preserves the order in which particles appear in the DataFrame.
sim_trajs_pred_method1_list = [
    # Sort each trajectory chronologically, select the required columns,
    # and convert the resulting DataFrame to a NumPy array.
    trajectory.sort_values("frame")[["frame", "x", "y"]].to_numpy()

    # Iterate over all particle trajectories.
    for _, trajectory in sim_trajs_pred_method1.groupby(
        "particle",
        sort=False,
    )

    # Discard trajectories detected in fewer than the required number
    # of distinct frames.
 
   if trajectory["frame"].nunique() >= min_trajectory_length
]

Create a video with overlayed localizations and trajectories.

In [ ]:
sim_video_method1_results = utils.make_video_with_trajs(
    trajs_pred_list=sim_trajs_pred_method1_list,
    video=sim_video,
    fov_size=sim_image_size,
    trajs_gt_list=sim_trajs_gt_list,
    figure_title="Simulated video"
)

# Display the video.
sim_video_method1_results

### Evaluating Linking Performance

Evaluate the end-to-end performance performance of the tracking pipeline (detection + linking) using the following metrics:

- **F1 score:** Harmonic mean of precision and recall, summarizing the balance between missed trajectories and spurious detections..

- **TP (True Positives):** Number of ground-truth trajectories correctly matched to a predicted trajectory.

- **FP (False Positives):** Number of predicted trajectories that cannot be matched to any ground-truth trajectory (spurious tracks).

- **FN (False Negatives):** Number of ground-truth trajectories for which no corresponding predicted trajectory is found (missed tracks).

- **α:** Global tracking accuracy measuring the agreement between matched ground-truth and predicted trajectories, while ignoring unmatched (spurious) predicted trajectories. This metric reflects how well correctly predicted trajectories are reconstructed.

- **β:** A stricter global accuracy metric that extends α by penalizing unmatched (spurious) trajectories, thus providing a more conservative and realistic assessment of overall tracking performance.

Detailed mathematical definitions and matching criteria are provided in [Chenouard et al., Nature Methods, 2014](https://www.nature.com/articles/nmeth.2808), which serves as the reference benchmark for particle-tracking evaluation.


In [ ]:
# Evaluate performance metrics.
utils.trajectory_metrics(
    sim_trajs_gt_list,
    sim_trajs_pred_method1_list,
    eps=5,
);

Display the reconstructed trajectories together with their ground truth.

In [ ]:
#  Compute the total squared distance between all trajectories to match
#  predicted trajectories with ground truth.
matched_pairs, _, _ = utils.trajectory_assignment(
    sim_trajs_gt_list,
    sim_trajs_pred_method1_list,
    eps=5,
)

# Plot the trajectories.
utils.plot_trajectory_matches(
    sim_trajs_gt_list, sim_trajs_pred_method1_list, matched_pairs,
)

Also display a trajectory timeline plot that visualizes the temporal support of each trajectory and its frame-level matching outcome. 
Paired ground-truth trajectories are shown in black, whereas ground-truth trajectories with no matched prediction (false negatives) are shown in purple.

Matched predicted trajectories are represented by short horizontal segments, with color encoding the matching status at each frame:
-   Blue: correct detections (prediction present and within the spatial threshold of the ground truth).
-   Red: localization errors (prediction present but farther than the threshold from the ground truth).
-   Orange: extra detections occurring during ground-truth blinking or absence.

Predicted trajectories with no corresponding ground truth (false positives) are shown in gray.

In [ ]:
# Plot trajectory timeline from matches.
utils.plot_trajectory_timeline(
    sim_trajs_gt_list,
    sim_trajs_pred_method1_list,
    matched_pairs,
)

Calculate the time-averaged MSD for all the trajectories and compare curves obtained for matching trajectories (same color).

In [ ]:
utils.plot_TAMSDs(
    trajs_pred=sim_trajs_pred_method1_list,
    trajs_gt=sim_trajs_gt_list,
    matched_pairs=matched_pairs,
) 

### Linking Localizations in Experiments

Apply the same steps to track the experiment and visualize the results.

In [ ]:
# Link detections across frames into trajectories using trackpy.link().
exp_trajs_pred_method1 = tp.link(
    df_exp_video,
    search_range=5, 
    memory=3,
)

# Minimum number of distinct frames required to keep a trajectory.
min_trajectory_length = 15

# Group detections by particle, discard short trajectories, and convert
# the remaining trajectories to chronologically ordered NumPy arrays.
exp_trajs_pred_method1_list = [
    # Sort each trajectory by frame, select the required columns,
    # and convert the resulting DataFrame to a NumPy array.
    trajectory.sort_values("frame")[["frame", "x", "y"]].to_numpy()

    # Iterate over all particle trajectories.
    # `sort=False` preserves the order in which particles appear.
    for _, trajectory in exp_trajs_pred_method1.groupby(
        "particle",
        sort=False,
    )

    # Keep only trajectories observed in at least the required number
    # of distinct frames.
    if trajectory["frame"].nunique() >= min_trajectory_length
]

exp_video_method1_results = utils.make_video_with_trajs(
    trajs_pred_list=exp_trajs_pred_method1_list,
    video=exp_video,
    fov_size=exp_image_size,
)

# Display the video.
exp_video_method1_results

Calculate the time-averaged MSD for the trajectories.

In [ ]:
utils.plot_TAMSDs(trajs_pred=exp_trajs_pred_method1_list)

## Method 2: Linear Assignment Problem (LAP) with LapTrack


LapTrack solves the trajectory linking problem by formulating it as a Linear Assignment Problem (LAP), a well-established optimization approach in multi-object tracking. It builds a cost matrix that quantifies dissimilarity—typically based on spatial distance—between particle detections in consecutive frames. Lower distances correspond to lower costs and indicate higher likelihoods of correspondence.

LapTrack uses the Hungarian algorithm to solve this assignment problem efficiently, minimizing the total cost across the matrix. This allows it to determine the globally optimal set of assignments across frames, enabling robust trajectory reconstruction even under challenging conditions such as high particle density or noisy detections.

Examples and tutorials using LapTrack are available in the [LapTrack documentation](https://github.com/yfukai/laptrack/tree/main/docs/examples).

### Linking Localizations in Simulations

Apply the method to the localization dataframe.

In [ ]:
# Configure the linear-assignment tracker.

# Maximum displacement, in pixels, allowed between detections in
# consecutive frames.
max_linking_distance = 5

# Maximum number of missing frames that can be bridged.
max_missing_frames = 3

# Maximum displacement, in pixels, allowed when closing a trajectory gap.
# This can be larger than the frame-to-frame distance because the particle
# has had more time to move.
max_gap_closing_distance = max_linking_distance * np.sqrt(max_missed_frames)

# LapTrack uses squared Euclidean distance by default. Therefore, the distance
# thresholds must be squared before they are passed as cost cutoffs.
laptrack = LapTrack(
    metric="sqeuclidean",
    cutoff=max_linking_distance**2,
    gap_closing_metric="sqeuclidean",
    gap_closing_cutoff=max_gap_closing_distance**2,
    gap_closing_max_frame_count=max_missing_frames,
)

# Link detections across frames.

# The first returned DataFrame contains the tracked detections. The other two
# outputs describe splitting and merging events, which are not used here.
sim_trajs_pred_method2, _, _ = laptrack.predict_dataframe(
    df=df_sim_video,
    coordinate_cols=["x", "y"],
    only_coordinate_cols=True,
)

# Move the frame, detection index, and tracking identifiers from the
# DataFrame index into regular columns.
sim_trajs_pred_method2 = sim_trajs_pred_method2.reset_index()

Create a trajectory list from the output dataframe.

In [ ]:
# Minimum number of distinct frames required to keep a trajectory.
min_trajectory_length = 10

# Group detections by track ID, discard short trajectories, and convert
# the remaining trajectories to chronologically ordered NumPy arrays.
sim_trajs_pred_method2_list = [
    # Sort each trajectory by frame, select the required columns,
    # and convert the resulting DataFrame to a NumPy array.
    trajectory.sort_values("frame")[["frame", "x", "y"]].to_numpy()

    # Iterate over all trajectories.
    # `sort=False` preserves the order in which track IDs first appear.
    for _, trajectory in sim_trajs_pred_method2.groupby(
        "track_id",
        sort=False,
    )

    # Keep only trajectories observed in at least the required number
    # of distinct frames.
    if trajectory["frame"].nunique() >= min_trajectory_length
]

Create a video with overlayed localizations and trajectories.

In [ ]:
sim_video_method2_results = utils.make_video_with_trajs(
    trajs_pred_list=sim_trajs_pred_method2_list,
    video=sim_video,
    fov_size=sim_image_size,
    trajs_gt_list=sim_trajs_gt_list,
)

# Display the video.
sim_video_method2_results

### Evaluating Linking Performance

Evaluate the overall performance of the tracking.

In [ ]:
# Evaluate performance metrics.
utils.trajectory_metrics(
    sim_trajs_gt_list,
    sim_trajs_pred_method2_list,
    eps=5,
);

Display the reconstructed trajectories together with the ground truth.

In [ ]:
# Compute the total squared distance between all trajectories to match
# predicted trajectories with ground truth.
matched_pairs, _, _ = utils.trajectory_assignment(
    sim_trajs_gt_list,
    sim_trajs_pred_method2_list,
    eps=5,
)

# Plot the trajectories.
utils.plot_trajectory_matches(
    sim_trajs_gt_list, sim_trajs_pred_method2_list, matched_pairs,
)

Display a trajectory timeline plot.

In [ ]:
# Plot trajectory timeline from matches.
utils.plot_trajectory_timeline(
    sim_trajs_gt_list,
    sim_trajs_pred_method2_list,
    matched_pairs,
)

Calculate the time-averaged MSD for all the trajectories and compare curves obtained for matching trajectories (same color).

In [ ]:
utils.plot_TAMSDs(
    trajs_pred=sim_trajs_pred_method2_list,
    trajs_gt=sim_trajs_gt_list,
    matched_pairs=matched_pairs,
) 

### Linking Localizations in Experiments

Apply the same steps to track the experiment and visualize the results.

In [ ]:
# Link detections across frames into trajectories using LapTrack.
exp_trajs_pred_method2, _, _ = laptrack.predict_dataframe(
    df_exp_video,
    ["x", "y"],
    only_coordinate_cols=True,
)

# Convert the frame and track identifiers from index levels into columns.
exp_trajs_pred_method2 = exp_trajs_pred_method2.reset_index()

# Minimum number of distinct frames required to keep a trajectory.
min_trajectory_length = 15

# Group detections by track ID, discard short trajectories, and convert
# the remaining trajectories to chronologically ordered NumPy arrays.
exp_trajs_pred_method2_list = [
    # Sort each trajectory by frame, select the required columns,
    # and convert the resulting DataFrame to a NumPy array.
    trajectory.sort_values("frame")[["frame", "x", "y"]].to_numpy()

    # Iterate over all trajectories.
    # `sort=False` preserves the order in which track IDs first appear.
    for _, trajectory in exp_trajs_pred_method2.groupby(
        "track_id",
        sort=False,
    )

    # Keep only trajectories observed in at least the required number
    # of distinct frames.
    if trajectory["frame"].nunique() >= min_trajectory_length
]

sim_video_method2_results = utils.make_video_with_trajs(
    trajs_pred_list=exp_trajs_pred_method2_list,
    video=exp_video,
    fov_size=exp_image_size,
)

# Display the video.
sim_video_method2_results

Calculate the time-averaged MSD for the trajectories.

In [ ]:
utils.plot_TAMSDs(trajs_pred=exp_trajs_pred_method2_list)

## Method 3: MAGIK

MAGIK is a framework designed to analyze the motion of dynamic systems, including cells, bacteria, individual molecules, colloids, and other active particles. The name stands for Motion Analysis through Graph Neural Network Inductive Knowledge.

At its core, MAGIK uses graph neural networks (GNNs) to learn patterns in particle movement and to infer trajectories across frames. This data-driven approach enables MAGIK to outperform traditional tracking methods in challenging conditions, such as dense particle fields, complex interaction dynamics, or non-Brownian motion.

Thanks to its ability to learn and generalize motion priors, MAGIK is particularly effective in noisy or ambiguous experimental settings, making it a strong complement—or even an alternative—to classical tools like TrackPy and LapTrack.

For more details, see the publication [Geometric Deep Learning Reveals the Spatiotemporal Features of Microscopic Motion](https://www.nature.com/articles/s42256-022-00595-0) *Nat Mach Intell* **5**, 71–82 (2023).

### Training MAGIK with Simulations

To train MAGIK effectively, you first need to generate appropriate training data. The experimental dataset in this case features single proteins undergoing heterogeneous random motion. In first approximation, you'll simply consider Brownian motion and you'll use the `simulate_Brownian_trajs()` function to produce groups of synthetic trajectories. Particle blinking is also simulated by introducing a probability of dropping points in the trajectory.

**Note:** It is crucial that the simulated training data accurately reflect the motion characteristics of your experimental particles. If your system exhibits pure diffusion (Brownian motion), the training simulations should mirror that. Conversely, if your experimental data involve additional dynamics—such as drift, confinement, or driven flow (e.g. in nanofluidic systems)—these should be incorporated into the training data to ensure MAGIK learns the correct motion priors.

In [ ]:
# Parameters for training dataset.
train_dataset_size = 256  # Number of videos
train_number_particles = 20  # Number of particles per video
train_number_timesteps = 30  # Number of frames per video
train_fov_size = 64  # Size of the field of view


# Initiate a dataframe containing all the simulated trajectories.
df_train_dataset = []
for video_index in range(train_dataset_size):
    # Simulate trajectories for one video.
    sim_trajs_train_dataset = utils.simulate_Brownian_trajs(
        num_particles=train_number_particles,
        num_timesteps=train_number_timesteps,
        fov_size=train_fov_size,
        diffusion_std=(0.1, 1.0),  # Corresponds to sqrt(2Dt)
    )

    sim_trajs_train_dataset = utils.apply_blinking(sim_trajs_train_dataset)

    sim_trajs_train_dataset_list = utils.trajs_array_to_list(
        trajs=sim_trajs_train_dataset,
    )

    # Make into dataframe with columns centroid-0, centroid-1, (positions), 
    # frame (which frame in the video), label (which particle in that frame),
    # and set (which video).
    for traj_index, traj in enumerate(sim_trajs_train_dataset_list):
        df_traj = pd.DataFrame(
            traj[:, 1:],
            columns=["centroid-0", "centroid-1"],
        )
        df_traj["frame"] = traj[:, 0].astype(int)
        df_traj["label"] = traj_index
        df_traj["set"] = f"{video_index}"
        df_train_dataset.append(df_traj)

# Concatenate to dataframe.
df_train_dataset = pd.concat(df_train_dataset, ignore_index=True)

Normalize the trajectory coordinates between 0 and 1 by dividing for the fov size.

In [ ]:
# Normalize centroids between 0 and 1.
norm_factor = np.array([train_fov_size, train_fov_size])

df_train_dataset.loc[:, df_train_dataset.columns.str.contains("centroid")] = (
    df_train_dataset.loc[
        :,
        df_train_dataset.columns.str.contains("centroid")
    ] / norm_factor
)

To train MAGIK with the simulated trajectories, you need to produce a graph representation with the function `GraphFromTrajectories()`, defined in the utility file `utils.py`. If particle density or the `max_frame_distance` are high, this might take a few minutes.

In [ ]:
# Instantiate the graph constructor.
graph_constructor = utils.GraphFromTrajectories(
    connectivity_radius=5/train_fov_size,
    max_frame_distance=max_missed_frames+1,
)

# Generate graph from training data using graph constructor.
train_dataset_graph = graph_constructor(df=df_train_dataset)

Define the augmentation pipeline and the dataloader.

In [ ]:
# Initialize the graph dataset class.
# `Dt` is the time difference between frames to sample from the graph.
# Specify augmentations with transform.
train_dataset = utils.GraphDataset(
    train_dataset_graph,
    dataset_size=256,
    Dt=4,
    transform=Compose(
        [
            utils.RandomRotation(),
            utils.RandomFlip(),
        ]
    )
)

# Initialize the training data loader.
train_loader = DataLoader(
    train_dataset,
    batch_size=24,
    shuffle=True,
    drop_last=True,
)

Define the architecture and its hyperparameters.

In [ ]:
magik = dl.GraphToEdgeMAGIK(
    [96,] * 4,1,
    out_activation=torch.nn.Identity,
)

magik.encoder[0].configure(
    hidden_features=[32, 64],
    out_features=96,
    out_activation=torch.nn.ReLU,
)

magik.encoder[1].configure(
    hidden_features=[32, 64],
    out_features=96,
    out_activation=torch.nn.ReLU,
)

magik.head.configure(hidden_features=[64, 32]);


classifier_magik = dl.BinaryClassifier(
    model=magik,
    optimizer=dl.Adam(lr=1e-3),
    loss = utils.NodewiseNLLLoss(),
).build()

classifier_magik.model.replace("selector", dl.FromDict("edge_attr", "edge_index", "x"))
classifier_magik.model.replace("head", utils.SoftmaxHead(classifier_magik.model.head))
classifier_magik.train_metrics = MetricCollection({})

trainer_magik = dl.Trainer(max_epochs=10, accelerator="auto")

Train the model or load pre-existing weights.

In [ ]:
# Define whether to force training even if weights exist.
force_training = True

# Define folder and model path.
folder_name = "MAGIK"
magik_model_path = os.path.join(folder_name, "magik_weights.pth")

# Train or load weights.
if not os.path.isfile(magik_model_path) or force_training:
    print("Training MAGIK model (either forced or weights not found).")

    # Ensure save directory exists.
    os.makedirs(folder_name, exist_ok=True)

    # Train the model.
    trainer_magik.fit(classifier_magik, train_loader)

    # Plot training history.
    trainer_magik.history.plot()
    
    # Save trained weights.
    torch.save(magik.state_dict(), magik_model_path)
    print(f"Saved MAGIK weights to '{magik_model_path}'.")
else:
    # Load pre-trained weights.
    magik.load_state_dict(torch.load(magik_model_path, weights_only=True))
    print(f"Loaded preexisting MAGIK weights from '{magik_model_path}'.")

# Set the model to evaluation mode.
classifier_magik.eval();

# Transfer the model to the best available device (optional).
classifier_magik.to(device);

### Evaluate MAGIK on a test video

Generate test data using the same pipeline used for the training set.

In [ ]:
# Parameters for test dataset.
test_number_particles = 30  # Number of particles per video
test_number_timesteps = 50  # Number of frames per video
test_fov_size = 64  # Size of the field of view

# Initiate a dataframe containing all the simulated trajectories.
df_test_dataset = []

# Simulate trajectories for one video.
sim_trajs_test_dataset = utils.simulate_Brownian_trajs(
    num_particles=test_number_particles,
    num_timesteps=test_number_timesteps,
    fov_size=test_fov_size,
    diffusion_std=0.5,
)

sim_trajs_test_dataset = utils.apply_blinking(sim_trajs_test_dataset)

sim_trajs_test_dataset_list = utils.trajs_array_to_list(
    trajs=sim_trajs_test_dataset,
)

# Make into dataframe with columns centroid-0, centroid-1, (positions), 
# frame (which frame in the video), label (which particle in that frame),
# and set (which video).
for traj_index, traj in enumerate(sim_trajs_test_dataset_list):
    df_traj = pd.DataFrame(
        traj[:, 1:],
        columns=["centroid-0", "centroid-1"],
    )
    df_traj["frame"] = traj[:, 0].astype(int)
    df_traj["label"] = traj_index
    df_traj["set"] = f"{video_index}"
    df_test_dataset.append(df_traj)


# Concatenate to dataframe.
df_test_dataset = pd.concat(df_test_dataset, ignore_index=True)

Normalize the trajectory coordinates between 0 and 1 by dividing for the fov size.

In [ ]:
# Normalize centroids between 0 and 1.
norm_factor = np.array([test_fov_size, test_fov_size])

df_test_dataset.loc[:, df_test_dataset.columns.str.contains("centroid")] = (
    df_test_dataset.loc[
        :,
        df_test_dataset.columns.str.contains("centroid")
    ] / norm_factor
)

Produce a graph representation.

In [ ]:
# Generate graph from test data using graph constructor.
graph_constructor = utils.GraphFromTrajectories(
    connectivity_radius=5/test_fov_size,
    max_frame_distance=max_missed_frames+1,
)

test_dataset_graph = graph_constructor(df=df_test_dataset)

Use the trained model to obtain predictions.

In [ ]:
# Perform prediction on test graph.
sim_trajs_edges_pred_method3,edge_index, probs_all, all_src, dummy_mask = classifier_magik(test_dataset_graph[0].to(device))

predictions = (sim_trajs_edges_pred_method3, edge_index, probs_all, all_src, dummy_mask)

Transform the model output into trajectories using `ComputeTrajectories()` and convert them into a list.

In [ ]:
# Instantiate trajectory constructor.
trajectory_constructor = utils.ComputeTrajectories(p_min=0.1)

# Reconstruct trajectories from predicted edges.
sim_trajs_pred_method3 = trajectory_constructor(
    test_dataset_graph[0].cpu(),
    predictions,
)

# Keep trajectories observed in at least min_trajectory_length frames.
min_trajectory_length = 10

sim_trajs_pred_method3 = [
    trajectory
    for trajectory in sim_trajs_pred_method3
    if len(trajectory) >= min_trajectory_length
]

# Convert the predicted trajectories to a list format.
sim_trajs_pred_method3_list = utils.make_list(
    sim_trajs_pred_method3, test_dataset_graph[0], test_fov_size,
)

Evaluate model performance on the test video.

In [ ]:
# Evaluate performance metrics.
utils.trajectory_metrics(
    sim_trajs_test_dataset_list,
    sim_trajs_pred_method3_list,
    eps=5,
);

### Linking Localizations in Simulations

You'll use the trajectories in the simulated video as the test dataset for the trained model of MAGIK. First, format the simulated dataframe. Then, convert the localizations corresponding to the simulated trajectories into a graph.

In [ ]:
# Rename for compatibility with label format.
df_sim_video_formatted = df_sim_video.rename(
    columns={"x": "centroid-0", "y": "centroid-1"}
)

# Add label, set, and solution columns.
df_sim_video_formatted[["label", "set"]] = 0

# Normalize coordinates to [0, 1].
frame_height, frame_width, _ = sim_video[0].shape

df_sim_video_formatted.loc[:, df_sim_video_formatted.columns.str.contains("centroid")] = (
    df_sim_video_formatted.loc[
        :,
        df_sim_video_formatted.columns.str.contains("centroid")
    ] / sim_image_size
)

graph_constructor = utils.GraphFromTrajectories(
    connectivity_radius=3/sim_image_size,
    max_frame_distance=max_missed_frames,
)

# Generate a graph from graph_constructor. As test_graph returns a list of
#  graphs, we select the first element from the list as it only has 1 element.
sim_video_graph = graph_constructor(df=df_sim_video_formatted)[0]

# Transfer graph to the best available device (optional).
sim_video_graph = sim_video_graph.to(device)

Apply the trained model of MAGIK to predict the edge features.

In [ ]:
# Perform prediction on test graph.
sim_trajs_edges_pred_method3, edge_index, probs_all, all_src, dummy_mask = classifier_magik(sim_video_graph)

predictions = (sim_trajs_edges_pred_method3, edge_index, probs_all, all_src, dummy_mask)

The edge feature can be used to obtain the predicted trajectories using the dedicated class.

In [ ]:
sim_trajs_pred_method3 = trajectory_constructor(
    sim_video_graph.cpu(),
    predictions,
)

# Keep trajectories observed in at least min_trajectory_length frames.
# min_trajectory_length = 15

sim_trajs_pred_method3 = [
    trajectory
    for trajectory in sim_trajs_pred_method3
    if len(trajectory) >= min_trajectory_length
]

# Convert the predicted trajectories to a list format.
sim_trajs_pred_method3_list = utils.make_list(
    sim_trajs_pred_method3, sim_video_graph, sim_image_size,
)

Create a video with overlayed localizations and trajectories.

In [ ]:
# Create a video with the predicted trajectories.
sim_video_method3_results = utils.make_video_with_trajs(
    trajs_pred_list=sim_trajs_pred_method3_list,
    video=sim_video,
    fov_size=sim_image_size,
    trajs_gt_list=sim_trajs_gt_list,
)

# Plot the video.
sim_video_method3_results

### Evaluating Linking Performance

Evaluate the overall performance of the tracking.

In [ ]:
# Evaluate performance metrics.
utils.trajectory_metrics(
    sim_trajs_gt_list,
    sim_trajs_pred_method3_list,
    eps=5,
);

Display the reconstructed trajectories together with the ground truth.

In [ ]:
#  Compute the total squared distance between all trajectories to match
#  predicted trajectories with ground truth.
matched_pairs, _, _ = utils.trajectory_assignment(
    sim_trajs_gt_list,
    sim_trajs_pred_method3_list,
    eps=5,
)

# Plot the trajectories.
utils.plot_trajectory_matches(
    sim_trajs_gt_list, sim_trajs_pred_method3_list, matched_pairs,
)

Display a trajectory timeline plot.

In [ ]:
# Plot trajectory timeline from matches.
utils.plot_trajectory_timeline(
    sim_trajs_gt_list,
    sim_trajs_pred_method3_list,
    matched_pairs,
)

Calculate the time-averaged MSD for all the trajectories and compare curves obtained for matching trajectories (same color).

In [ ]:
utils.plot_TAMSDs(
    trajs_pred = sim_trajs_pred_method3_list,
    trajs_gt = sim_trajs_gt_list,
    matched_pairs = matched_pairs,
) 

### Linking Localizations in Experiments

Apply the same steps to track the experiment and visualize the results.

In [ ]:
# Rename for compatibility with label format.
df_exp_video_formatted = df_exp_video.rename(
    columns={"x": "centroid-0", "y": "centroid-1"}
)

# Add label, set, and solution columns.
df_exp_video_formatted[["label", "set", "solution"]] = 0

# Normalize coordinates to [0, 1].
frame_height, frame_width, _ = sim_image.shape
df_exp_video_formatted[["centroid-0", "centroid-1"]] /= [frame_width,
                                                         frame_height]


# Generate a graph from graph_constructor. As test_graph returns a list of
#  graphs, we select the first element from the list as it only has 1 element.
exp_video_graph = graph_constructor(df=df_exp_video_formatted)[0].to(device)

# Perform prediction on graph.
exp_trajs_edges_pred_method3, edge_index, probs_all, all_src, dummy_mask = classifier_magik(exp_video_graph)
predictions = (exp_trajs_edges_pred_method3, edge_index, probs_all, all_src, dummy_mask)


exp_trajs_pred_method3 = trajectory_constructor(
    exp_video_graph.cpu(),
    predictions,
)

# Keep trajectories observed in at least min_trajectory_length frames.
exp_trajs_pred_method3 = [
    trajectory
    for trajectory in exp_trajs_pred_method3
    if len(trajectory) >= min_trajectory_length
]

# Convert the predicted trajectories to a list format.
exp_trajs_pred_method3_list = utils.make_list(
    exp_trajs_pred_method3, exp_video_graph, exp_image_size,
)

# Create a video with the predicted trajectories.
exp_video_method3_results = utils.make_video_with_trajs(
    trajs_pred_list=exp_trajs_pred_method3_list,
    video=exp_video,
    fov_size=exp_image_size,
)

exp_video_method3_results

Calculate the time-averaged MSD for the trajectories.

In [ ]:
utils.plot_TAMSDs(trajs_pred=exp_trajs_pred_method3_list)

## Linking statistics

Generate 100 videos with groung truth for quantitative comparisson of linking methods

In [ ]:
# Simulation parameters.
number_particles = 10
number_timesteps = 30
num_simulations = 10
sim_size = sim_image_size 

sim_trajs_gt_results = []
sim_trajs_gt_list_results = []
sim_video_results = []


# Dictionary for particle properties. Dimensions are set with lambda
# functions to introduce variety to the dataset.
particle_properties = {
    "intensity": lambda: np.random.uniform(1.0, 1.6) * 1E3,
    "z": lambda: np.random.uniform(-500, 500) * dt.units.nm,
    "radius": particle_radius * dt.units.nm,

# Set the optical properties of the microscope.  This dictionary is a DeepTrack
# optics object.
optics_properties = dt.Fluorescence(
    NA=1.47,  # Numerical aperture
    wavelength=580 * dt.units.nm,
    refractive_index_medium=1.33,
    output_region=[0, 0, sim_crop_size, sim_crop_size],
    magnification=lambda: np.random.uniform(1.0, 2.0),
    resolution=pixel_size_nm * dt.units.nm,  # Camera effective resolution
)

# Noise statistics
background_props={"peak_snr": 100, "background_mean": 117, "background_std": 3.5}

for n_simulation in range(num_simulations):
    print(f"Simulation {n_simulation}")
    # Simulate trajectories for one video.
    sim_trajs_gt = utils.simulate_Brownian_trajs(
        num_particles=number_particles,
        num_timesteps=number_timesteps,
        fov_size=sim_size,
        diffusion_std=0.5,  # Corresponds to sqrt(2Dt)
    )

    # Break trajectories going in/out of FOV.
    sim_trajs_gt_list = utils.trajs_array_to_list(
        trajs=sim_trajs_gt,
    )

    # Simulate video.
    sim_video = utils.transform_to_video(
        np.delete(sim_trajs_gt, 2, 2),  # Remove frame axis
        fov_size=sim_size,
        core_particle_props=sphere_properties,
        optics_props=optics_properties,
        background_props=background_props,
        save_video=False,
    )

    sim_trajs_gt_results.append(sim_trajs_gt)
    sim_trajs_gt_list_results.append(sim_trajs_gt_list)
    sim_video_results.append(sim_video)

Save results of simulations

In [ ]:
from pathlib import Path

# =============================================================================
# Save simulation results
# =============================================================================

output_dir = Path("simulation_outputs")
output_dir.mkdir(parents=True, exist_ok=True)

# # Videos should all have the same shape.
# sim_video_array = np.stack(
#     [np.asarray(video) for video in sim_video_results],
#     axis=0,
# )

# # Ground-truth trajectories may have variable shapes.
# sim_trajs_gt_array = np.empty(
#     len(sim_trajs_gt_results),
#     dtype=object,
# )

# sim_trajs_gt_list_array = np.empty(
#     len(sim_trajs_gt_list_results),
#     dtype=object,
# )

# for i in range(len(sim_trajs_gt_results)):
#     sim_trajs_gt_array[i] = sim_trajs_gt_results[i]
#     sim_trajs_gt_list_array[i] = sim_trajs_gt_list_results[i]

# np.save(
#     output_dir / "sim_video_results.npy",
#     sim_video_array,
# )

# np.save(
#     output_dir / "sim_trajs_gt_results.npy",
#     sim_trajs_gt_array,
#     allow_pickle=True,
# )

# np.save(
#     output_dir / "sim_trajs_gt_list_results.npy",
#     sim_trajs_gt_list_array,
#     allow_pickle=True,
# )

# print("Simulation results saved.")

Load simulations if available

In [ ]:
# =============================================================================
# Load simulation results
# =============================================================================

sim_video_results = np.load(
    output_dir / "sim_video_results.npy",
)

sim_trajs_gt_results = np.load(
    output_dir / "sim_trajs_gt_results.npy",
    allow_pickle=True,
)

sim_trajs_gt_list_results = np.load(
    output_dir / "sim_trajs_gt_list_results.npy",
    allow_pickle=True,
)

print("Simulation results loaded.")

In [ ]:
utils.play_video(sim_video_results[0])

### Detection with LodeSTAR
Perform detections on all videos